# Question B：使用 Funnelling Approach 进行特征选择

本题使用沪深 300 指数数据构造机器学习特征，并通过 filter、wrapper 和 embedded 三类方法逐步收窄特征集合。

## 方法概览

本研究把特征选择设计成三层漏斗：

1. **Filter 方法**：先在模型外部检查数据质量、删除高度相关特征，并用互信息衡量单个特征与目标变量的关联。
2. **Wrapper 方法**：使用时间序列交叉验证，比较不同特征数量下的模型 ROC AUC，选择在训练集交叉验证中表现最稳的候选特征组。
3. **Embedded 方法**：在候选特征组上训练 XGBoost，用 gain importance 排序，再用时间序列交叉验证决定最终保留多少个重要特征。

这样做的目标不是寻找“单个最强特征”，而是在金融短期预测这种低信噪比任务中，保留一组弱但互补的预测信号。

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

# 允许 notebook 从项目根目录或 Exam3 目录运行。
if Path.cwd().name != "Exam3" and (Path.cwd() / "Exam3").exists():
    os.chdir(Path.cwd() / "Exam3")
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from workflow_draft import (
    WorkflowConfig,
    build_features,
    train_test_split_time,
    feature_selection_funnel,
)

config = WorkflowConfig(data_path=Path("CSI300_2005_2026.csv"))
config

WorkflowConfig(data_path=WindowsPath('CSI300_2005_2026.csv'), start_date='2010-01-01', target_threshold=0.0015, test_size=0.2, random_state=42, cv_splits=5, cv_gap=1)

## 数据、标签与样本切分

研究对象为沪深 300 指数。为避免未来信息泄露，所有特征均使用当日或过去窗口的数据计算；目标变量使用下一交易日收益率。

标签定义为：若下一交易日对数收益率大于 **0.15%**，则记为 1，否则记为 0。这个阈值用于把极小的近零上涨归为非显著上涨，避免把市场噪声机械地视作有效上涨信号。

样本从 2010 年开始，最后 20% 作为时间顺序上的测试集。第 2 题的特征选择只使用训练集完成。

In [ ]:
data, feature_cols = build_features(config)
X_train, X_test, y_train, y_test, train_frame, test_frame = train_test_split_time(
    data, feature_cols, config
)

summary = pd.DataFrame({
    "项目": [
        "完整样本起止", "训练集起止", "测试集起止",
        "完整样本数", "训练集样本数", "测试集样本数",
        "候选特征数", "训练集正类比例", "测试集正类比例", "标签阈值",
    ],
    "数值": [
        f"{data.index.min().date()} 至 {data.index.max().date()}",
        f"{train_frame.index.min().date()} 至 {train_frame.index.max().date()}",
        f"{test_frame.index.min().date()} 至 {test_frame.index.max().date()}",
        len(data), len(train_frame), len(test_frame), len(feature_cols),
        round(y_train.mean(), 4), round(y_test.mean(), 4), config.target_threshold,
    ],
})
summary

,项目,数值
0,完整样本起止,2010-01-04 至 2026-05-18
1,训练集起止,2010-01-04 至 2023-02-02
2,测试集起止,2023-02-03 至 2026-05-18
3,完整样本数,3899
4,训练集样本数,3119
5,测试集样本数,780
6,候选特征数,75
7,训练集正类比例,0.4431
8,测试集正类比例,0.4179
9,标签阈值,0.0015


## 原始候选特征池

原始特征池覆盖收益、趋势、波动率、成交量、K 线形态和技术指标。特征数量必须足够多，才能让后续漏斗式选择有意义；同时，所有滚动窗口都只使用历史数据。

In [ ]:
feature_groups = pd.DataFrame({
    "类别": ["收益与动量", "趋势/均线", "波动率与极值", "成交量", "K 线结构", "技术指标", "日历变量"],
    "示例特征": [
        "log_ret_1, ret_sum_2, ret_sum_3, ret_sum_10, ret_sum_120",
        "ma_ratio_3, ma_ratio_5, ma_ratio_20, ma_ratio_120",
        "volatility_20, rolling_min_ret_5, rolling_max_ret_60, atr_20",
        "volume_ret, volume_z_10, volume_z_20, volume_z_40",
        "range_pct, gap_ret, upper_shadow, lower_shadow, close_pos",
        "rsi_6, rsi_14, rsi_21, macd, macd_signal, macd_hist",
        "dow",
    ],
})
feature_groups

,类别,示例特征
0,收益与动量,"log_ret_1, ret_sum_2, ret_sum_3, ret_sum_10, r..."
1,趋势/均线,"ma_ratio_3, ma_ratio_5, ma_ratio_20, ma_ratio_120"
2,波动率与极值,"volatility_20, rolling_min_ret_5, rolling_max_..."
3,成交量,"volume_ret, volume_z_10, volume_z_20, volume_z_40"
4,K 线结构,"range_pct, gap_ret, upper_shadow, lower_shadow..."
5,技术指标,"rsi_6, rsi_14, rsi_21, macd, macd_signal, macd..."
6,日历变量,dow


## 第一步：Filter 方法

Filter 阶段不依赖最终模型。这里先删除训练集中两两相关系数绝对值大于 0.98 的冗余特征，再用 mutual information 对剩余特征排序。高度相关的特征通常携带重复信息，保留它们会增加模型复杂度，也会让特征重要性解释变得不稳定。互信息则可捕捉非线性关联，比简单线性相关更适合树模型前的粗筛。

In [ ]:
selection = feature_selection_funnel(X_train, y_train, config)

print(f"原始特征数: {len(feature_cols)}")
print(f"高相关删除数: {len(selection['corr_dropped'])}")
print(f"高相关删除后特征数: {len(selection['corr_features'])}")
print()
print("被删除的高相关特征:")
print(selection["corr_dropped"])

原始特征数: 75
高相关删除数: 11
高相关删除后特征数: 64

被删除的高相关特征:
['simple_ret_1', 'body_pct', 'ret_mean_2', 'ma_ratio_2', 'ret_mean_3', 'ret_mean_5', 'ret_mean_10', 'ret_mean_20', 'ret_mean_40', 'ret_mean_60', 'ret_mean_120']


In [ ]:
mi_top20 = selection["mi_scores"].head(20).rename("mutual_information").to_frame()
mi_top20

,mutual_information
ret_sum_10,0.018311
ret_sum_3,0.016329
rolling_max_ret_40,0.014959
volatility_60,0.012091
ma_ratio_3,0.012012
ret_sum_40,0.011584
volatility_120,0.010612
log_ret_1,0.010156
ret_sum_60,0.009404
dow,0.008289


## 第二步：Wrapper 方法

Wrapper 阶段把特征选择放回模型训练流程中。具体做法是：按互信息排名依次取前 N 个特征，并用 `TimeSeriesSplit` 计算 XGBoost 在训练集交叉验证中的 ROC AUC。

这里使用时间序列交叉验证，而不是随机 K 折，是因为金融时间序列存在明显的时间顺序。随机打乱会让训练数据间接看到未来结构，产生过于乐观的结果。

In [ ]:
wrapper_table = selection["wrapper_table"].copy()
wrapper_table["cv_mean_auc"] = wrapper_table["cv_mean_auc"].round(4)
wrapper_table["cv_std_auc"] = wrapper_table["cv_std_auc"].round(4)
wrapper_table

,n_features,cv_mean_auc,cv_std_auc
0,10,0.5602,0.0266
1,15,0.5579,0.0264
2,20,0.5583,0.0321
3,25,0.5579,0.0269
4,30,0.5553,0.0190
5,35,0.5530,0.0192
6,40,0.5559,0.0263
7,50,0.5589,0.0296
8,64,0.5614,0.0262


从上表可以看到，训练集交叉验证并没有支持过度压缩特征数量；保留高相关筛选后的全部 64 个特征时，平均 ROC AUC 最高。这个结果符合短期市场方向预测的特点：单个特征信号很弱，多个弱信号组合后才可能形成较稳定的信息。

## 第三步：Embedded 方法

Embedded 阶段使用 XGBoost 的 gain importance。Gain 衡量某个特征被用于树分裂时带来的平均损失下降，更接近模型内部实际使用特征的方式。

为避免只看一次模型的重要性排序而过拟合，最终仍然用时间序列交叉验证比较 top K 重要特征的表现。

In [ ]:
gain_top25 = selection["gain_scores"].head(25).rename("xgb_gain_importance").to_frame()
gain_top25

,xgb_gain_importance
ma_ratio_5,0.038572
macd,0.031022
rolling_max_ret_60,0.029655
rolling_min_ret_2,0.023588
ret_sum_3,0.023397
rolling_min_ret_3,0.022184
ret_sum_2,0.021541
ma_ratio_3,0.021138
rolling_max_ret_2,0.021038
volume_z_20,0.019846


In [ ]:
embedded_table = selection["embedded_table"].copy()
embedded_table["cv_mean_auc"] = embedded_table["cv_mean_auc"].round(4)
embedded_table["cv_std_auc"] = embedded_table["cv_std_auc"].round(4)
embedded_table

,n_features,cv_mean_auc,cv_std_auc
0,10,0.5405,0.0254
1,15,0.5686,0.0332
2,20,0.5656,0.0273
3,25,0.5648,0.0290
4,30,0.5651,0.0231
5,35,0.5660,0.0273
6,40,0.5668,0.0292
7,64,0.5610,0.0291


Embedded 阶段最终选择 top 40 个特征。该数量在训练集时间序列交叉验证中给出最高平均 ROC AUC，同时相比 64 个候选特征明显降低了维度。

In [ ]:
final_features = selection["final_features"]
final_feature_table = pd.DataFrame({
    "序号": range(1, len(final_features) + 1),
    "最终特征": final_features,
    "XGBoost gain": selection["gain_scores"].loc[final_features].round(6).values,
})
final_feature_table

,序号,最终特征,XGBoost gain
0,1,ma_ratio_5,0.038572
1,2,macd,0.031022
2,3,rolling_max_ret_60,0.029655
3,4,rolling_min_ret_2,0.023588
4,5,ret_sum_3,0.023397
5,6,rolling_min_ret_3,0.022184
6,7,ret_sum_2,0.021541
7,8,ma_ratio_3,0.021138
8,9,rolling_max_ret_2,0.021038
9,10,volume_z_20,0.019846


## 特征选择结论

最终保留的特征主要集中在短期均线偏离、近期极端收益、短期动量、成交量标准化、MACD/RSI、价格区间和跳空收益等维度。它们共同描述了市场在最近数日到数月内的趋势、反转、波动和成交状态。

这组特征将作为第 3 题梯度提升分类模型的输入。